# Anion-binding ESP calibration (illustrative reproduction)

This notebook sketches an ESP→log(Ka) correlation workflow using placeholder training data bundled with suprakit.

Literature anchor: Priyadarshini *et al.*, *JACS Au* **2025** — replace the DOI below with the published article DOI when citing formally.


In [ ]:
# Run with: make notebook
from suprakit.core.device import detect_device

print(detect_device())


In [ ]:
from suprakit.anionfit.receptors import build_resorcin4arene

codes = ["Br", "CHO", "NO2", "CN"]
mols = [build_resorcin4arene([c] * 4, upper_rim="hydroxyl") for c in codes]
len(mols)


## ESP values

`anionfit.esp.compute_esp_at_bowl_center` is intentionally deferred pending verified `tblite` ESP extraction APIs.

Use placeholder ESP inputs from `priyadarshini_2025.csv` until ESP evaluation lands.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from suprakit.anionfit.predict import ESPLogKaModel

model = ESPLogKaModel.from_priyadarshini_2025()
esp = model.esp_train
truth = model.logka_train

grid = np.linspace(float(np.min(esp)), float(np.max(esp)), 200)
pred, lo, hi = model.predict_with_ci(grid)

plt.figure(figsize=(6, 4))
plt.scatter(esp, truth, label="training (placeholder)")
plt.plot(grid, pred, label="OLS")
plt.fill_between(grid, lo, hi, alpha=0.2)
plt.xlabel("ESP (a.u., placeholder)")
plt.ylabel("log10 Ka")
plt.legend()
plt.tight_layout()


In [ ]:
from suprakit.anionfit.substituents import EWG_SMILES, enumerate_patterns

patterns = enumerate_patterns(list(EWG_SMILES.keys()), symmetric_only=False)
len(patterns)


In [ ]:
import numpy as np
import pandas as pd
from importlib.resources import files
from IPython.display import display

from suprakit.anionfit.predict import ESPLogKaModel
from suprakit.anionfit.substituents import EWG_SMILES, enumerate_patterns

model = ESPLogKaModel.from_priyadarshini_2025()
patterns = enumerate_patterns(list(EWG_SMILES.keys()), symmetric_only=False)

csv_path = files("suprakit.anionfit.data") / "priyadarshini_2025.csv"
df = pd.read_csv(csv_path, comment="#")
lookup = dict(zip(df["substituent"], df["esp_au"]))

rows = []
for pat in patterns[:30]:
    esp_mean = float(np.mean([lookup[c] for c in pat]))
    rows.append(
        {
            "pattern": "+".join(pat),
            "esp_mean_placeholder": esp_mean,
            "logKa_pred": float(model.predict(np.array([esp_mean]))[0]),
        }
    )

ranked = pd.DataFrame(rows).sort_values("logKa_pred", ascending=False).head(10)
display(ranked)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(ranked["pattern"], ranked["logKa_pred"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("predicted log10 Ka (placeholder)")
plt.tight_layout()


## Limitations

- Linearity is a modeling choice, not a physical law.
- Implicit solvation, conformational ensembles, and probe-point definitions dominate uncertainty.
